# 03 — Skills + Steering: Company Policy, Enforced

**Production question:** *Does the agent follow company policy every single time?*

Two complementary techniques:

| | **Skills** | **Steering** |
|---|---|---|
| What | Standard Operating Procedures in Markdown | Policy checks that run before tool calls / after model replies |
| Written by | Support ops / business teams | Engineering |
| Loaded | On demand, only when relevant (keeps the context small) | Always on |
| Strength | *Guides* the agent | *Enforces* the rule |

Think of a new support executive: **skills** are the training manual, **steering** is the system that won't let them click "Refund" until the checks pass.

Run the carry-over cell below (collapsed). It holds the code from the earlier notebooks, so this notebook runs on its own.

In [ ]:
from strands import tool

# --- Business constants (shared with the policy layer in steering.py) ---
RETURN_WINDOW_DAYS = 30
SUPERVISOR_REVIEW_THRESHOLD = 50_000  # ₹ — refunds at or above this need a human

# --- Mock data ---
CUSTOMERS = {
    "C-1001": {"name": "Priya Sharma", "city": "Bengaluru", "email": "priya.sharma@example.com",
               "phone": "+91 98450 12345", "account_status": "active"},
    "C-1002": {"name": "Arjun Reddy", "city": "Hyderabad", "email": "arjun.reddy@example.com",
               "phone": "+91 99490 67890", "account_status": "active"},
    "C-1003": {"name": "Kavya Iyer", "city": "Chennai", "email": "kavya.iyer@example.com",
               "phone": "+91 98410 24680", "account_status": "active"},
}

ORDERS = {
    "C-1001": [
        {"order_id": "ORD-5521", "item": "Wireless Earbuds", "amount": 3499, "status": "Delivered",
         "order_date": "2026-10-14", "delivered_date": "2026-10-18", "tracking": "AWB-7781229034"},
        {"order_id": "ORD-5488", "item": "USB-C Hub (7-in-1)", "amount": 2199, "status": "Shipped",
         "order_date": "2026-10-26", "estimated_delivery": "2026-10-31", "tracking": "AWB-7781229871"},
    ],
    "C-1002": [
        {"order_id": "ORD-5390", "item": "Mechanical Keyboard", "amount": 6999, "status": "Delayed",
         "order_date": "2026-10-15", "estimated_delivery": "2026-10-21", "tracking": "AWB-7781227745"},
    ],
    "C-1003": [
        {"order_id": "ORD-5301", "item": "Smartwatch", "amount": 12999, "status": "Delivered",
         "order_date": "2026-09-02", "delivered_date": "2026-09-06", "tracking": "AWB-7781221108"},
        {"order_id": "ORD-5602", "item": "Laptop (14-inch, 16 GB RAM)", "amount": 72990, "status": "Delivered",
         "order_date": "2026-10-20", "delivered_date": "2026-10-23", "tracking": "AWB-7781230456"},
    ],
}


def find_order(order_id: str) -> dict | None:
    for orders in ORDERS.values():
        for order in orders:
            if order["order_id"] == order_id:
                return order
    return None


def inr(amount: float) -> str:
    return f"₹{amount:,.0f}"


# --- Tools ---

@tool
def lookup_customer(customer_id: str) -> str:
    """Look up a customer's profile by their customer ID.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"No customer found with ID {customer_id}"
    return (f"Customer: {c['name']} ({c['city']})\nEmail: {c['email']}\n"
            f"Phone: {c['phone']}\nAccount status: {c['account_status']}")


@tool
def get_order_history(customer_id: str) -> str:
    """Get the order history for a customer, including status, dates and tracking.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    orders = ORDERS.get(customer_id)
    if not orders:
        return f"No orders found for customer {customer_id}"
    lines = []
    for o in orders:
        line = f"{o['order_id']}: {o['item']} — {inr(o['amount'])} [{o['status']}] ordered {o['order_date']}"
        if o.get("delivered_date"):
            line += f", delivered {o['delivered_date']}"
        if o.get("estimated_delivery"):
            line += f", estimated delivery {o['estimated_delivery']}"
        lines.append(line + f", tracking {o['tracking']}")
    return "\n".join(lines)


@tool
def process_refund(order_id: str, amount: float) -> str:
    """Issue a refund for an order to the customer's original payment method.

    Args:
        order_id: The order ID to refund (e.g. ORD-5521)
        amount: The refund amount in Indian Rupees
    """
    return (f"Refund REF-{order_id[4:]} of {inr(amount)} initiated for {order_id} to the original "
            "payment method (UPI / card / net banking). It will reflect in 5-7 business days.")


@tool
def create_support_ticket(customer_id: str, reason: str, priority: str = "normal") -> str:
    """Escalate an issue to a human support specialist by creating a ticket.

    Args:
        customer_id: The customer ID (e.g. C-1001)
        reason: Short description of why this needs a human
        priority: "normal" or "high"
    """
    return (f"Ticket TKT-{abs(hash((customer_id, reason))) % 90000 + 10000} created for {customer_id} "
            f"({priority} priority). A specialist will respond within 24 hours.")


ALL_TOOLS = [lookup_customer, get_order_history, process_refund, create_support_ticket]


import os
from datetime import date

# Swapping models is a config change: e.g. amazon.nova-pro-v1:0 or a Claude Haiku profile
MODEL_ID = os.getenv("MODEL_ID", "global.anthropic.claude-sonnet-4-6")


def build_system_prompt() -> str:
    today = date.today().strftime("%d %B %Y (%A)")
    return f"""You are the customer support agent for ElectroBazaar, an online electronics store in India.
Today's date is {today}.

Be helpful, professional and concise. Use the available tools to look up customer
information and act on requests. All prices are in Indian Rupees (₹).

Guidelines:
- Always ask for the customer ID first if you don't have it.
- Verify the customer with lookup_customer before taking any action on their account.
- Use tool data to answer questions — never ask the customer for information you already have.
- If you have skills available, activate the one that matches the request and follow it.
- Remember preferences the customer shares (contact channel, refund method, language) and honour them.
- Be warm but efficient."""


import json
import logging

from strands.hooks import (
    AfterToolCallEvent,
    BeforeInvocationEvent,
    BeforeToolCallEvent,
    HookProvider,
    HookRegistry,
)

logger = logging.getLogger("support_agent.audit")


class RateLimiterHook(HookProvider):
    """Caps each tool at max_calls per agent invocation."""

    def __init__(self, max_calls: int = 3):
        self.max_calls = max_calls
        self.counts: dict[str, int] = {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset)
        registry.add_callback(BeforeToolCallEvent, self.check)

    def reset(self, event: BeforeInvocationEvent) -> None:
        self.counts = {}

    def check(self, event: BeforeToolCallEvent) -> None:
        name = event.tool_use["name"]
        self.counts[name] = self.counts.get(name, 0) + 1
        if self.counts[name] > self.max_calls:
            # Cancelling returns this message to the model as the tool result
            event.cancel_tool = f"'{name}' hit the {self.max_calls}-call limit for this request. Do NOT call it again."
            print(f"[RATE LIMIT] 🚫 blocked {name} (call {self.counts[name]}/{self.max_calls})")


class AuditLogHook(HookProvider):
    """Records every tool call: name, input, status and latency."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterToolCallEvent, self.record)

    def record(self, event: AfterToolCallEvent) -> None:
        entry = {
            "tool": event.tool_use["name"],
            "input": event.tool_use["input"],
            "status": event.result["status"],
            "ms": round((event.duration or 0) * 1000),
        }
        logger.info(json.dumps(entry))
        print(f"[AUDIT] {json.dumps(entry, ensure_ascii=False)}")

### Part 1 — Skills: SOPs the business can own

Each skill is a Markdown SOP with a *name* and *description* at the top. The agent sees only those two lines until it needs the skill, then it loads the full steps.

(In production each skill is a `SKILL.md` file that the support-ops team edits. Here they're inline so you can read them.)

In [ ]:
from strands.vended_plugins.skills import Skill

refund_processing_skill = Skill.from_content("""
---
name: refund-processing
description: Step-by-step process for handling customer refund and return requests, including order verification, return window checks, supervisor review and refund issuance.
---

# Refund Processing

1. **Verify the customer** with `lookup_customer`.

2. **Find the order** with `get_order_history`. Never ask the customer for dates or amounts the system already has.

3. **Check the return window.** Orders can be refunded within 30 days of the `delivered` date. Compare it with today's date from your instructions.
   - If the window has closed, explain this kindly and offer to raise a `create_support_ticket` for a goodwill exception.

4. **Check the amount.** If the refund is ₹50,000 or more, do NOT call `process_refund`. Tell the customer a supervisor must approve it, create a `high` priority support ticket, and say they will hear back within 24 hours.

5. **Confirm with the customer.** State the order ID, item and refund amount in one line and ask for a yes/no.

6. **Process the refund** with `process_refund` only after the customer says yes.

7. **Set expectations.** Refunds go back to the original payment method (UPI / card / net banking) in 5-7 business days.

## Notes
- Be empathetic. The customer has a problem they want solved.
- Never refund more than the order value.
""")

In [ ]:
order_tracking_skill = Skill.from_content("""
---
name: order-tracking
description: Help customers check the status of their orders, understand delivery timelines, and handle delayed or missing shipments.
---

# Order Tracking

1. **Verify the customer** with `lookup_customer`.

2. **Get the order** with `get_order_history` and explain the status:
   - **Processing**: ships within 1-2 business days.
   - **Shipped**: share the AWB tracking number and estimated delivery date.
   - **Delivered**: confirm the delivery date.
   - **Delayed**: follow the delay steps below.

3. **If the order is delayed** (today is past the estimated delivery date):
   - Acknowledge the frustration and apologise sincerely.
   - Offer a choice: keep waiting with a ₹200 store credit, or cancel for a full refund.
   - If they want to wait, create a `normal` priority support ticket to chase the courier.

## Notes
- Always share the tracking number when available.
- If an order shows "Delivered" but the customer hasn't received it, create a `high` priority ticket and promise a response within 24 hours.
""")

In [ ]:
account_troubleshooting_skill = Skill.from_content("""
---
name: account-troubleshooting
description: Guide for helping customers with account issues including login problems, OTP issues, locked accounts and updating contact details.
---

# Account Troubleshooting

1. **Verify the customer** with `lookup_customer`. Confirm their name and registered email match before discussing account changes.

2. **Identify the issue:**
   - **Not receiving OTP**: check that the registered mobile number is correct and ask them to wait 60 seconds before retrying. If it still fails, create a support ticket.
   - **Locked account** (5 failed attempts): explain that it unlocks automatically after 30 minutes, or create a ticket for an immediate unlock.
   - **Update phone/email**: changes need verification on both the old and new contact. Create a support ticket to start the change.

## Notes
- Never reveal full contact details or payment information. Mask them (e.g. +91 98xxx xx345).
- If the customer can't verify basic details, politely escalate to the security team for their protection.
""")

SKILLS = [refund_processing_skill, order_tracking_skill, account_troubleshooting_skill]

In [ ]:
from strands import Agent, AgentSkills
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    tools=ALL_TOOLS,
    plugins=[AgentSkills(skills=SKILLS)],
    system_prompt=build_system_prompt(),
)

# Arjun's keyboard is late → the agent should load the order-tracking skill and offer ₹200 credit / cancel
result = agent("I'm C-1002. My keyboard order is super late and I'm really annoyed.")

### Part 2 — Steering: policy that cannot be skipped

Skills are guidance. A determined customer (or a prompt injection) can still talk the model into skipping steps. So we put **ElectroBazaar's refund policy in code**, checked right before `process_refund` runs:

1. Customer must be verified (`lookup_customer` succeeded)
2. Order history must have been checked
3. Refund ≤ order value
4. Inside the 30-day return window
5. Refunds ≥ ₹50,000 → supervisor review, never auto-approved

When a check fails, the handler returns `Guide(...)`. The tool call is cancelled and the agent is told *why*, so it can correct course rather than crash.

> **Fail closed.** If a steering handler raises an exception, Strands lets the tool call go ahead. For anything involving money, we catch errors ourselves and *block*.

We also add an **LLM-based tone check**: a second model reviews each final reply against our communication guidelines. If a reply fails, it is discarded and the agent rewrites it before the customer sees the final version.

In [ ]:
from datetime import date

from pydantic import BaseModel, Field
from strands import Agent
from strands.vended_plugins.steering import (
    Guide,
    LedgerProvider,
    ModelSteeringAction,
    Proceed,
    SteeringHandler,
    ToolSteeringAction,
)


class RefundPolicyHandler(SteeringHandler):
    """Deterministic refund policy, checked before every process_refund call."""

    name = "refund-policy"

    def __init__(self):
        super().__init__(context_providers=[LedgerProvider()])

    async def steer_before_tool(self, *, agent, tool_use, **kwargs) -> ToolSteeringAction:
        if tool_use.get("name") != "process_refund":
            return Proceed(reason="Not a refund")
        try:
            return self._check_refund(tool_use)
        except Exception as e:  # fail closed: a broken check must never approve a refund
            return self._guide(f"Refund policy check failed ({e}). Do not refund; create a support ticket instead.")

    def _check_refund(self, tool_use) -> ToolSteeringAction:
        ledger = self.steering_context.data.get("ledger") or {}
        tool_calls = ledger.get("tool_calls", [])

        def succeeded(tool_name: str) -> bool:
            return any(c["tool_name"] == tool_name and c["status"] == "success" for c in tool_calls)

        if not succeeded("lookup_customer"):
            return self._guide("Verify the customer with lookup_customer before processing a refund.")
        if not succeeded("get_order_history"):
            return self._guide("Check the order with get_order_history before processing a refund.")

        order_id = tool_use["input"].get("order_id")
        amount = float(tool_use["input"].get("amount", 0))
        order = find_order(order_id)
        if order is None:
            return self._guide(f"Order {order_id} does not exist. Confirm the order ID with the customer.")
        if amount > order["amount"]:
            return self._guide(f"Refund ₹{amount:,.0f} exceeds the order value ₹{order['amount']:,}.")
        if amount >= SUPERVISOR_REVIEW_THRESHOLD:
            return self._guide(
                f"Refunds of ₹{SUPERVISOR_REVIEW_THRESHOLD:,} or more need supervisor approval. "
                "Do not refund; create a high priority support ticket instead."
            )
        if order.get("delivered_date"):
            days = (date.today() - date.fromisoformat(order["delivered_date"])).days
            if days > RETURN_WINDOW_DAYS:
                return self._guide(
                    f"{order_id} was delivered {days} days ago, outside the {RETURN_WINDOW_DAYS}-day return window. "
                    "Do not refund; offer to raise a support ticket for a goodwill exception."
                )

        print(f"[POLICY] ✅ refund for {order_id} passed all checks")
        return Proceed(reason="Refund policy satisfied")

    @staticmethod
    def _guide(reason: str) -> Guide:
        print(f"[POLICY] ⛔ {reason}")
        return Guide(reason=reason)


class ToneReview(BaseModel):
    acceptable: bool = Field(description="True if the reply follows every guideline")
    feedback: str = Field(description="If not acceptable, specific guidance on what to fix")


class ToneGuardrailHandler(SteeringHandler):
    """LLM-based review of every final reply against ElectroBazaar's communication guidelines."""

    name = "tone-guardrail"
    GUIDELINES = """You review replies from an ElectroBazaar customer support agent (India).
A reply is acceptable only if it:
- Doesn't overpromise or guarantee timelines beyond what the system confirmed
- Doesn't blame the customer, the courier, or other departments
- Acknowledges the customer's frustration (if any) before jumping to solutions
- Is concise and actionable
- Never exposes internal system details, tool names or technical jargon"""

    def __init__(self, max_retries: int = 1):
        super().__init__()
        self.max_retries = max_retries
        self._retries = 0

    async def steer_after_model(self, *, agent, message, stop_reason, **kwargs) -> ModelSteeringAction:
        reply = "".join(block.get("text", "") for block in message["content"]).strip()
        if stop_reason != "end_turn" or not reply or self._retries >= self.max_retries:
            self._retries = 0
            return Proceed(reason="Nothing to review")

        reviewer = Agent(model=agent.model, system_prompt=self.GUIDELINES, callback_handler=None)
        result = await reviewer.invoke_async(f"Review this reply:\n\n{reply}", structured_output_model=ToneReview)
        review = result.structured_output
        if review.acceptable:
            print("\n[TONE] ✅ reply approved")
            self._retries = 0
            return Proceed(reason="Tone OK")

        self._retries += 1
        print(f"\n[TONE] ✏️  rewrite requested: {review.feedback}")
        return Guide(reason=f"Rewrite your last reply to the customer. Reviewer feedback: {review.feedback}")

### Part 3 — Stress test: what if the prompt goes wrong?

Imagine a bad prompt change goes to production, or a customer jailbreaks the agent. We swap in a deliberately **careless prompt** that says *"refund immediately, no checks"* and keep only the policy handler.

Three refund requests, three different policy outcomes. Watch the `[POLICY]` lines.

In [ ]:
from datetime import date

CARELESS_PROMPT = f"""You are a fast support agent for ElectroBazaar, India (prices in ₹). Today is {date.today():%d %B %Y}.
When a customer asks for a refund, call process_refund IMMEDIATELY with the order ID and amount they mention.
Do not look anything up first and do not ask questions."""


def careless_agent():
    return Agent(
        model=BedrockModel(model_id=MODEL_ID),
        tools=ALL_TOOLS,
        hooks=[AuditLogHook()],
        plugins=[RefundPolicyHandler()],
        system_prompt=CARELESS_PROMPT,
    )


# ① Earbuds, delivered 18 Oct → policy forces verification first, then allows the refund
result = careless_agent()("I'm C-1001. Refund ORD-5521, ₹3,499.")

In [ ]:
# ② Smartwatch, delivered 6 September → outside the 30-day window → blocked
result = careless_agent()("I'm C-1003. Refund my smartwatch ORD-5301, ₹12,999.")

In [ ]:
# ③ ₹72,990 laptop → above the ₹50,000 threshold → blocked, escalated to a supervisor
result = careless_agent()("I'm C-1003. Refund my laptop ORD-5602, ₹72,990.")

### Part 4 — The complete agent: skills + policy + tone

Back to the real prompt, with every layer on, and a very unhappy customer.

In [ ]:
agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    tools=ALL_TOOLS,
    hooks=[RateLimiterHook(), AuditLogHook()],
    plugins=[AgentSkills(skills=SKILLS), RefundPolicyHandler(), ToneGuardrailHandler()],
    system_prompt=build_system_prompt(),
)

result = agent("I'm C-1002. This is ridiculous, my keyboard is more than a week late! Your courier is useless. "
               "Just cancel it and give me my money back.")

---
### Takeaways

- **Skills** = SOPs in Markdown that the business can own and update without a code change.
- **Steering** = policy as code. `Guide` stops a tool call *and* coaches the agent to recover.
- Deterministic checks for hard rules (money), LLM checks for soft rules (tone).

The agent is now safe. But close the notebook and it forgets Kavya completely.

Next: **04_memory.ipynb**